<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 34px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; What can PATSTAT actually prove?</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        Forty answers were guessed. Eleven of them are matters of record &mdash; so let us go and check, and see <strong>how much the number moves</strong>.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; <a href="https://patentreports.depa.tech" target="_blank"
           style="color: #be0f05; text-decoration: none; font-weight: 600;">created by Arne Kr&uuml;ger</a>
        &nbsp;&middot;&nbsp; model: <strong>EPO IPScore 3.01</strong>
        &nbsp;&middot;&nbsp; scenario analysis after Riccardo Priore's NPV Target Planner
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 680px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What this notebook does</strong>
            <br/>Step&nbsp;1 &nbsp;&middot;&nbsp; The map: what is reachable, and what it can change
            <br/>Step&nbsp;2 &nbsp;&middot;&nbsp; Find the patent family
            <br/>Step&nbsp;3 &nbsp;&middot;&nbsp; <strong>A1 &middot; is it granted &mdash; and was it opposed?</strong>
            <br/>Step&nbsp;4 &nbsp;&middot;&nbsp; A3 &middot; how much term is left, and are the fees paid?
            <br/>Step&nbsp;5 &nbsp;&middot;&nbsp; <strong>A5 &middot; designated in 38 states &mdash; in force in how many?</strong>
            <br/>Step&nbsp;6 &nbsp;&middot;&nbsp; A7 &middot; are disputes customary in this field?
            <br/>Step&nbsp;7 &nbsp;&middot;&nbsp; E1 / E2 / E7 &middot; the applicant's own footprint
            <br/>Step&nbsp;8 &nbsp;&middot;&nbsp; The proxies &mdash; and the ones not good enough to count
            <br/>Step&nbsp;9 &nbsp;&middot;&nbsp; <strong>The diff: what the adviser guessed vs what the record says</strong>
            <br/>Step&nbsp;10 &nbsp;&middot;&nbsp; Hand the corrected answers to notebooks 3 and 4
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 680px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Part 2 of 4 &mdash; the <strong>only</strong> notebook that needs EPO&nbsp;TIP.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            It queries <strong>PATSTAT&nbsp;PROD</strong> and cannot run anywhere else, which is why
            it ships <em>without stored outputs</em> while notebooks&nbsp;1,&nbsp;3 and&nbsp;4 ship
            executed. Run order: <strong>1 &rarr; 2 &rarr; 3 &rarr; 4</strong>.
        </div>
    </div>
</div>

---

## Step 1 — The map, and the uncomfortable part of it

Before querying anything, it is worth being precise about what this notebook can and cannot
achieve — because the honest answer is surprising, and it is better to say it at the start than
to let a reader discover it at the end.

Two facts about the IPScore questionnaire, both computed below rather than asserted:

1. **Eleven of the forty questions can be spoken to by a PATSTAT query.** Three of them
   strongly, four usefully, three only as labelled proxies, one as context.
2. **Eight of the forty questions carry money** — they are the only ones that reach the Net
   Present Value at all.

Now put those two sets side by side. Run the cell.

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

import ipscore_kit as kit
from ipscore_kit import Answer

spec = kit.load_spec()
example = kit.load_worked_example()
first_pass = example["scores"]

reachable = set(kit.PATSTAT_CANDIDATES)
money = {q.id for q in spec.oek_questions}

print(f"reachable by a query : {sorted(reachable)}")
print(f"carry money          : {sorted(money)}")
print(f"\nboth                 : {sorted(reachable & money) or 'NONE - the two sets do not overlap'}")
print(f"\n{len(reachable)} reachable, {len(money)} carry money, {len(reachable & money)} do both.")

reachable by a query : ['A1', 'A3', 'A4', 'A5', 'A7', 'B1', 'B2', 'C4', 'E1', 'E2', 'E7']
carry money          : ['B5', 'C2', 'C3', 'C6', 'D1', 'D2', 'D3', 'D4']

both                 : NONE - the two sets do not overlap

11 reachable, 8 carry money, 0 do both.


### Zero overlap. Read that again.

**Not one of the eleven questions PATSTAT can answer is one of the eight that move the number.**
Everything this notebook is about to measure — is the patent granted, was it opposed, how much
term is left, which countries it is actually in force in, whether it sits in the applicant's core
technology — changes the **profile**, the risk map and the conversation. It changes the Net
Present Value by **exactly zero euros**.

That is not a defect in this notebook. It is a property of the model, and it is the most useful
thing module 8 has to say:

> The parts of a patent valuation that are **checkable** and the parts that are **decisive** are
> disjoint sets. The money rests entirely on eight forecasts about a market — how fast it grows,
> how long the technology lives, what share of turnover it adds — and PATSTAT has nothing to say
> about any of them.

So what is the evidence layer *for*? Three things, none of which is moving the NPV:

* **It stops you being wrong about the record.** Whatever the queries below return, they return
  it with a date and an event code attached, and several of the adviser's confident answers will
  not survive contact with them. The number does not care; a client signing a licence would.
* **It makes the valuation auditable.** An answer with a date and an event code behind it can be
  checked by someone who does not trust you.
* **It shows where the guessing actually is.** Once eight answers are marked as the only movers
  and none of them is measurable, everybody in the room understands what kind of object a
  patent valuation is.

---

## Step 2 — Find the patent family

Our worked example is **`EP3074539B1`** — *"Method for detecting and characterising a
microorganism"*, **Q-Linea AB**, Uppsala. It came out of module 6's antibiotic-resistance corpus
(decision **V5**), so the two modules describe the same field.

Everything downstream keys off its `docdb_family_id`. We take it from `worked_example.json` but
re-derive the family members from PATSTAT rather than trusting the file.

> **A note on the queries in this notebook.** They were written against the schema established
> in `9_documentation/results-tipsession.md`, on a session that confirmed the legal-event tables
> exist and are populated. Five traps are accounted for: `granted` is `'Y'`/`'N'` and not
> a boolean, claim counts live on `tls211_pat_publn` and not `tls201_appln`,
> `tls803.event_impact` is `NULL` for every code, the lapsed state sits in
> `tls231.lapse_country` rather than `event_text`, and `tls231` has **ten** date columns but no
> plain `event_date` — the fifth one only surfaced when these queries first met the database.
>
> When a column does not exist, BigQuery does not say which one. It returns a generic
> *"Standard SQL dialect is currently selected"* message that names nothing at all, so the fix is
> always to dump `SELECT * … LIMIT 1` and read the real schema rather than re-reading the SQL.
>
> Each step still runs inside `measure()`, which catches a failing query, leaves that answer as
> `judgement` and carries on. A notebook that half-works and says which half is more useful in
> front of a room than one that stops on the first error.

In [2]:
from epo.tipdata.patstat import PatstatClient

patstat = PatstatClient(env='PROD')

def q(sql):
    """Run a query and return a DataFrame."""
    return pd.DataFrame(patstat.sql_query(sql, use_legacy_sql=False))

# Every measured answer is collected here: score, provenance, and the fact behind it.
evidence = {}
failures = {}

def measure(qid, fn):
    """Try to establish one answer from data. On failure, leave it as judgement and say so."""
    try:
        score, provenance, fact = fn()
        evidence[qid] = Answer(score, provenance=provenance, evidence=fact)
        moved = "" if score == first_pass[qid] else f"  (guess was {first_pass[qid]})"
        print(f"  {qid}  {first_pass[qid]} -> {score}  [{provenance}]{moved}\n      {fact}")
    except Exception as exc:
        failures[qid] = f"{type(exc).__name__}: {exc}"
        print(f"  {qid}  could not be measured - stays judgement\n      {failures[qid]}")

FAMILY_ID = example["patent"]["docdb_family_id"]

members = q(f"""
SELECT a.appln_id, a.appln_auth, a.appln_nr, a.appln_kind,
       a.appln_filing_date, a.earliest_filing_date, a.granted,
       a.docdb_family_size, a.nb_citing_docdb_fam,
       p.publn_auth, p.publn_nr, p.publn_kind, p.publn_date, p.publn_claims
FROM tls201_appln a
LEFT JOIN tls211_pat_publn p ON p.appln_id = a.appln_id
WHERE a.docdb_family_id = {FAMILY_ID}
ORDER BY a.appln_auth, p.publn_date
""")

print(f"family {FAMILY_ID}: {members['appln_id'].nunique()} applications, "
      f"{len(members)} publications, "
      f"authorities {sorted(members['appln_auth'].dropna().unique())}")
members.head(20)

family 53398085: 10 applications, 19 publications, authorities ['AU', 'CA', 'CN', 'EP', 'JP', 'KR', 'US', 'WO']


,appln_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_date,granted,docdb_family_size,nb_citing_docdb_fam,publn_auth,publn_nr,publn_kind,publn_date,publn_claims
0,473172207,AU,2015273443,A,2015-06-12,2014-06-13,Y,10,30,AU,2015273443,A1,2016-12-22,0
1,473172207,AU,2015273443,A,2015-06-12,2014-06-13,Y,10,30,AU,2015273443,B2,2021-03-04,0
2,474905039,CA,2949732,A,2015-06-12,2014-06-13,Y,10,30,CA,2949732,A1,2015-12-17,0
3,474905039,CA,2949732,A,2015-06-12,2014-06-13,Y,10,30,CA,2949732,C,2023-01-17,0
4,479480596,CN,201580031776,A,2015-06-12,2014-06-13,Y,10,30,CN,106661606,A,2017-05-10,0
5,479480596,CN,201580031776,A,2015-06-12,2014-06-13,Y,10,30,CN,106661606,B,2021-12-21,0
6,441666764,EP,15729146,A,2015-06-12,2014-06-13,Y,10,30,EP,3074539,A1,2016-10-05,0
7,441666764,EP,15729146,A,2015-06-12,2014-06-13,Y,10,30,EP,3074539,B1,2018-01-10,19
8,487005238,EP,17205276,A,2015-06-12,2014-06-13,Y,10,30,EP,3351642,A1,2018-07-25,21
9,487005238,EP,17205276,A,2015-06-12,2014-06-13,Y,10,30,EP,3351642,B1,2019-09-11,21


---

## Step 3 — A1 · Is it granted, and was it opposed?

The first question in the questionnaire, and one nobody should ever guess at. The five answers
run from *"patent not yet applied for"* to *"opposition period expired"* — so being granted is a
**4**, and surviving the nine-month opposition window untouched is a **5**.

Two sources, both facts:

* `tls201_appln.granted` — `'Y'` or `'N'`, not a boolean
* `tls231_inpadoc_legal_event` — the INPADOC codes `26N` *(no opposition filed)*, `26`
  *(opposition filed)*, `27O` *(rejected, patent survives)*, `27A` *(maintained in amended
  form)*, `27W` *(revoked)*

The last one matters most. A revoked patent is worth nothing, and no amount of expert judgement
about market growth changes that.

In [3]:
OPPOSITION_CODES = {
    "26N": "no opposition filed",
    "26":  "opposition filed",
    "27O": "opposition rejected - patent survives",
    "27A": "maintained in amended form",
    "27W": "patent REVOKED",
    "PLBP": "opposition withdrawn",
}

# tls231 has ten date columns and no plain `event_date`. The one that says when an event
# took legal effect is `event_effective_date` - but it carries 9999-12-31 as a "not
# applicable" sentinel, so fall back to the publication date whenever it does.
EVENT_DATE = ("CASE WHEN event_effective_date < DATE '9999-01-01' "
              "THEN event_effective_date ELSE event_publn_date END")


def a1_status():
    ep = members[(members["appln_auth"] == "EP")]
    granted = (ep["granted"] == "Y").any()

    ep_ids = tuple(int(i) for i in ep["appln_id"].unique())
    events = q(f"""
    SELECT event_code, MIN({EVENT_DATE}) AS first_seen
    FROM tls231_inpadoc_legal_event
    WHERE appln_id IN ({','.join(str(i) for i in ep_ids)})
      AND event_code IN ({','.join(repr(c) for c in OPPOSITION_CODES)})
    GROUP BY event_code
    """)
    seen = dict(zip(events["event_code"], events["first_seen"])) if len(events) else {}

    if "27W" in seen:
        return 2, "measured", f"EP patent REVOKED in opposition ({seen['27W']})"
    if not granted:
        return 3, "measured", "no EP grant recorded in PATSTAT"
    if "26N" in seen:
        return 5, "measured", (f"granted, and the opposition period expired with no opposition "
                               f"filed (26N, effective {seen['26N']})")
    if "26" in seen:
        outcome = ("maintained in amended form" if "27A" in seen else
                   "opposition rejected, patent survives" if "27O" in seen else "outcome pending")
        return 4, "measured", f"granted; opposition filed ({seen['26']}) - {outcome}"
    return 4, "measured", "granted; no opposition event recorded yet"

measure("A1", a1_status)

  A1  4 -> 5  [measured]  (guess was 4)
      granted, and the opposition period expired with no opposition filed (26N, effective 2018-10-11)


---

## Step 4 — A3 · How much term is left, and are the fees paid?

The five answers are pure arithmetic bands: 0–2 years, 2–4, 4–8, 8–12, more than 12. So the
*nominal* term is a lookup — earliest filing date plus twenty years — and there is no excuse for
guessing it.

But nominal term is an **upper bound**, and this is where the legal events earn their keep. A
patent whose renewal fees stopped being paid in year 8 has no term left at all, however far the
twenty years still runs. `PGFP` events carry `fee_renewal_year` and `fee_country`, so *"renewals
paid up to year N, in these countries"* is a fact rather than an assumption.

We report both, and score on the nominal band while stating the renewal position alongside —
because the band is what the questionnaire asks for.

In [4]:
from datetime import date

def a3_term():
    earliest = pd.to_datetime(members["earliest_filing_date"]).min()
    expiry = earliest + pd.DateOffset(years=20)
    years_left = (expiry - pd.Timestamp(date.today())).days / 365.25

    ep_ids = tuple(int(i) for i in members[members["appln_auth"] == "EP"]["appln_id"].unique())
    renewals = q(f"""
    SELECT fee_country, MAX(fee_renewal_year) AS paid_to_year, MAX(fee_payment_date) AS last_payment
    FROM tls231_inpadoc_legal_event
    WHERE appln_id IN ({','.join(str(i) for i in ep_ids)})
      AND event_code = 'PGFP'
    GROUP BY fee_country
    ORDER BY paid_to_year DESC
    """)

    # the questionnaire's own bands
    band = (1 if years_left < 2 else 2 if years_left < 4 else
            3 if years_left < 8 else 4 if years_left < 12 else 5)

    if len(renewals):
        top = int(renewals["paid_to_year"].max())
        where = ", ".join(sorted(renewals[renewals["paid_to_year"] == top]["fee_country"]))
        fee_note = f"; renewals paid to year {top} in {where}"
    else:
        fee_note = "; no renewal payments recorded"

    fact = (f"earliest filing {earliest:%Y-%m-%d}, nominal expiry {expiry:%Y-%m-%d} "
            f"= {years_left:.1f} years left{fee_note}")
    return band, "measured", fact

measure("A3", a3_term)

  A3  4 -> 3  [measured]  (guess was 4)
      earliest filing 2014-06-13, nominal expiry 2034-06-13 = 7.8 years left; renewals paid to year 11 in DE, FR, GB, SE


---

## Step 5 — A5 · Designated in how many states, in force in how many?

This is the step that justifies the whole notebook.

The question asks whether *"the patent's geographical coverage includes the relevant markets"*,
and the five answers run from *"a single national market"* to *"all existing and potentially
relevant market area countries"*. An adviser looking at a granted European patent naturally
reads the designation list and thinks: **most of Europe**.

The designation list is not the coverage. A European patent is granted for a set of designated
states, and then it **lapses in every one of them where the translation is not filed or the
renewal fee is not paid** — usually within a couple of years, usually in most of them. What
remains is a much smaller list, and only that list is protection.

Three sources, all in `tls231`:

* `designated_states` — what was designated at grant
* `lapse_country` / `lapse_date` / `lapse_text` — where it fell away (**not** `event_text`, which
  is empty)
* `PGFP` `fee_country` — where the fees are still being paid, i.e. where it is still alive

We also count the family's **national grants** — a US or Japanese patent is coverage just as
much as a German one — so the footprint is the surviving EP states *plus* those. Whether that
adds up to more or less than the adviser assumed is genuinely open until the query runs; what is
not open is that *designated* and *in force* are different numbers, and only one of them is
protection.

The provenance marker here is **`informed`**, not `measured`, and deliberately so: the footprint
is a matter of record, but which markets are the **relevant** ones is a business judgement no
query can make.

In [5]:
def a5_coverage():
    ep_ids = tuple(int(i) for i in members[members["appln_auth"] == "EP"]["appln_id"].unique())
    ids = ','.join(str(i) for i in ep_ids)

    designated = q(f"""
    SELECT designated_states FROM tls231_inpadoc_legal_event
    WHERE appln_id IN ({ids}) AND designated_states IS NOT NULL AND designated_states != ''
    LIMIT 5
    """)
    states = set()
    for row in designated.get("designated_states", []):
        states.update(s for s in str(row).replace(",", " ").split() if len(s) == 2)

    lapsed = q(f"""
    SELECT DISTINCT lapse_country FROM tls231_inpadoc_legal_event
    WHERE appln_id IN ({ids}) AND lapse_country IS NOT NULL AND lapse_country != ''
    """)
    lapse_set = set(lapsed["lapse_country"]) if len(lapsed) else set()

    alive = q(f"""
    SELECT fee_country, MAX(fee_renewal_year) AS paid_to
    FROM tls231_inpadoc_legal_event
    WHERE appln_id IN ({ids}) AND event_code = 'PGFP' AND fee_country IS NOT NULL
    GROUP BY fee_country
    """)
    in_force = set(alive["fee_country"]) - lapse_set if len(alive) else set()

    # national members of the family count too - a US or JP grant is coverage
    national = set(members[(members["granted"] == "Y") &
                           (~members["appln_auth"].isin(["EP", "WO"]))]["appln_auth"])
    footprint = in_force | national

    n = len(footprint)
    band = (1 if n <= 1 else 2 if n <= 5 else 3 if n <= 12 else 4 if n <= 25 else 5)
    fact = (f"designated in {len(states)} EPC states at grant, lapsed in {len(lapse_set)}; "
            f"in force in {sorted(in_force)}"
            + (f" plus national grants in {sorted(national)}" if national else "")
            + f" = {n} territories")
    return band, "informed", fact

measure("A5", a5_coverage)

  A5  3 -> 3  [informed]
      designated in 38 EPC states at grant, lapsed in 34; in force in ['DE', 'FR', 'GB', 'SE'] plus national grants in ['AU', 'CA', 'CN', 'JP', 'KR', 'US'] = 10 territories


---

## Step 6 — A7 · Are disputes customary in this field?

Note what A7 actually asks: not *"was this patent opposed"* but whether *"disputes and legal
proceedings are customary in the operative markets"*. That is a question about the
**neighbourhood**, not about our patent — so the evidence is an opposition *rate* across
comparable patents, which the `26N`/`26` pair gives directly: `26N` counts the grants that were
never opposed, `26` counts those that were.

The EP-wide baseline established on TIP is **≈ 4.5 %**. We compute the same rate restricted to
this patent's IPC subclasses and compare.

Marked **`informed`**: the rate is measured, but turning a percentage into *"customary"* uses
thresholds we chose, and those are a judgement. Stating them in the code is the least we can
do.

In [6]:
# our thresholds, written down so they can be argued with
RATE_BANDS = [(0.03, 5, "not customary"), (0.06, 4, "disputes exist"),
              (0.12, 3, "disputes are customary"), (0.20, 2, "legal proceedings exist"),
              (1.01, 1, "legal proceedings very customary")]

def a7_disputes():
    ipcs = q(f"""
    SELECT DISTINCT SUBSTR(ipc_class_symbol, 1, 4) AS subclass
    FROM tls209_appln_ipc
    WHERE appln_id IN ({','.join(str(int(i)) for i in members['appln_id'].unique())})
    """)
    subclasses = sorted(ipcs["subclass"].dropna().unique())

    rate = q(f"""
    SELECT COUNTIF(e.event_code = '26')  AS opposed,
           COUNTIF(e.event_code = '26N') AS not_opposed
    FROM tls231_inpadoc_legal_event e
    JOIN tls209_appln_ipc i ON i.appln_id = e.appln_id
    WHERE e.event_code IN ('26', '26N')
      AND SUBSTR(i.ipc_class_symbol, 1, 4) IN ({','.join(repr(s) for s in subclasses)})
    """)
    opposed = int(rate["opposed"].iloc[0])
    total = opposed + int(rate["not_opposed"].iloc[0])
    if total == 0:
        raise ValueError("no opposition events in this IPC neighbourhood")

    share = opposed / total
    band, label = next((b, l) for threshold, b, l in RATE_BANDS if share < threshold)
    fact = (f"{opposed:,} of {total:,} granted EP patents in {', '.join(subclasses)} were "
            f"opposed = {share:.1%} (EP baseline 4.5%) -> {label}")
    return band, "informed", fact

measure("A7", a7_disputes)

  A7  3 -> 3  [informed]
      22,141 of 330,611 granted EP patents in C12M, C12P, C12Q, G01N, G06T were opposed = 6.7% (EP baseline 4.5%) -> disputes are customary


---

## Step 7 — E1 / E2 / E7 · The applicant's own footprint

Three strategy questions that look like pure opinion and are not, because the applicant's own
filing history is on the record:

* **E1** *secure position in existing markets* — how much of this family sits in jurisdictions
  the applicant has filed in before
* **E2** *win new markets* — and how much of it is somewhere they have never been
* **E7** *part of the company's core technology* — the share of the applicant's whole portfolio
  in the same IPC subclass

All three are **`informed`**. The data says where the applicant files and what they file about;
whether that constitutes *"a very large degree"* of strategic intent is still a person's call.
The scale here is a 1–5 *degree*, which no query can pin down on its own.

In [7]:
def applicant_portfolio():
    """This family's applicant, and every other family they own."""
    ids = ','.join(str(int(i)) for i in members["appln_id"].unique())
    who = q(f"""
    SELECT p.person_id, p.psn_name, p.psn_sector
    FROM tls207_pers_appln pa
    JOIN tls206_person p ON p.person_id = pa.person_id
    WHERE pa.appln_id IN ({ids}) AND pa.applt_seq_nr > 0
    """)
    person_ids = ','.join(str(int(i)) for i in who["person_id"].unique())
    portfolio = q(f"""
    SELECT DISTINCT a.docdb_family_id, a.appln_auth,
           SUBSTR(i.ipc_class_symbol, 1, 4) AS subclass
    FROM tls207_pers_appln pa
    JOIN tls201_appln a ON a.appln_id = pa.appln_id
    LEFT JOIN tls209_appln_ipc i ON i.appln_id = a.appln_id
    WHERE pa.person_id IN ({person_ids}) AND pa.applt_seq_nr > 0
    """)
    return who, portfolio

who, portfolio = applicant_portfolio()
applicant_name = who["psn_name"].mode().iloc[0] if len(who) else "unknown"
n_families = portfolio["docdb_family_id"].nunique()
n_classified = portfolio.dropna(subset=["subclass"])["docdb_family_id"].nunique()
print(f"applicant: {applicant_name}  ({who['psn_sector'].mode().iloc[0] if len(who) else '?'})")
print(f"portfolio: {n_families} families, {portfolio['appln_auth'].nunique()} authorities"
      f"  ({n_classified} of them carry an IPC symbol - E7 can only count those)")

OUR_AUTH = set(members["appln_auth"].dropna())

def _degree(share):
    """Turn a share into one of the questionnaire's five 'degree' bands. Our thresholds."""
    return 1 if share < 0.10 else 2 if share < 0.30 else 3 if share < 0.55 else 4 if share < 0.80 else 5

def e1_existing():
    prior = set(portfolio[portfolio["docdb_family_id"] != FAMILY_ID]["appln_auth"].dropna())
    overlap = OUR_AUTH & prior
    share = len(overlap) / len(OUR_AUTH) if OUR_AUTH else 0
    return _degree(share), "informed", (
        f"{len(overlap)} of {len(OUR_AUTH)} jurisdictions in this family "
        f"({', '.join(sorted(overlap))}) are ones {applicant_name} had already filed in")

def e2_new():
    prior = set(portfolio[portfolio["docdb_family_id"] != FAMILY_ID]["appln_auth"].dropna())
    fresh = OUR_AUTH - prior
    share = len(fresh) / len(OUR_AUTH) if OUR_AUTH else 0
    return _degree(share), "informed", (
        f"{len(fresh)} of {len(OUR_AUTH)} jurisdictions are new for {applicant_name}"
        + (f" ({', '.join(sorted(fresh))})" if fresh else ""))

def e7_core():
    ours = set(portfolio[portfolio["docdb_family_id"] == FAMILY_ID]["subclass"].dropna())
    # only families carrying an IPC symbol can be compared; unclassified ones are not
    # evidence of anything either way, so they stay out of both halves of the fraction
    fam = portfolio.dropna(subset=["subclass"])
    total = fam["docdb_family_id"].nunique()
    same = fam[fam["subclass"].isin(ours)]["docdb_family_id"].nunique()
    share = same / total if total else 0
    return _degree(share), "informed", (
        f"{same} of the {total} {applicant_name} families that carry an IPC symbol "
        f"(out of {n_families} in total) are in the same subclass "
        f"({', '.join(sorted(ours))}) = {share:.0%}")

for qid, fn in [("E1", e1_existing), ("E2", e2_new), ("E7", e7_core)]:
    measure(qid, fn)

applicant: Q-LINEA  (COMPANY)
portfolio: 25 families, 9 authorities  (19 of them carry an IPC symbol - E7 can only count those)
  E1  4 -> 5  [informed]  (guess was 4)
      7 of 8 jurisdictions in this family (AU, CA, CN, EP, KR, US, WO) are ones Q-LINEA had already filed in
  E2  3 -> 2  [informed]  (guess was 3)
      1 of 8 jurisdictions are new for Q-LINEA (JP)
  E7  4 -> 4  [informed]
      12 of the 19 Q-LINEA families that carry an IPC symbol (out of 25 in total) are in the same subclass (C12M, C12P, C12Q) = 63%


---

## Step 8 — The proxies, and the ones not good enough to count

The last four candidates are weaker, and the discipline is to say so rather than to dress them
up. Two different outcomes here, and the difference between them is the point of the step.

**A4 · breadth of claim → claim count.** Claim *count* is not claim *breadth* — a single
sweeping claim beats twenty narrow ones. But the count is at least a real number with a real
benchmark: `tls211_pat_publn.publn_claims` is 100 % populated for granted EP `B1` documents, mean
**11.5 claims**. Take it from the `B1`, never the `A1` — the A-publication carries the *as-filed*
claims, which is a different claim set. We attach it as context and mark the answer `informed`.

**B1 / B2 · uniqueness and technical superiority → forward citations.** Here we stop. Citations
measure **attention**, not superiority: a heavily cited patent may be famous for being in
everyone's way. The count goes into the evidence field as context, and the provenance marker
**stays `judgement`**, because nothing about the number entitles us to change the score.

That distinction — *found data* is not the same as *the answer is now evidence* — is worth more
than either measurement.

**C4 · competitive products → the IPC neighbourhood.** Same treatment: context, not an answer.

In [8]:
EP_MEAN_CLAIMS = 11.5   # granted EP B1, filings 2010-2022 (established on TIP)

def a4_claims():
    b1 = members[(members["publn_auth"] == "EP") & (members["publn_kind"] == "B1")]
    b1 = b1.dropna(subset=["publn_claims"]).sort_values("publn_date")
    if b1.empty:
        raise ValueError("no EP B1 claim count - WO has 0% coverage and the A1 is the wrong set")
    # the earliest B1 is the parent; a later one is a divisional, which we mention but do not value
    n = int(b1["publn_claims"].iloc[0])
    band = 2 if n < 8 else 3 if n < 16 else 4
    return band, "informed", (f"{n} claims in the granted EP B1 vs a mean of {EP_MEAN_CLAIMS} "
                              f"for granted EP - a proxy for breadth, not a measure of it")

measure("A4", a4_claims)

# B1, B2, C4 - context only. The score is NOT changed and the marker stays judgement.
def context_only():
    fam_ids = ','.join(str(int(i)) for i in members["appln_id"].unique())
    citing = q(f"""
    SELECT COUNT(DISTINCT c.docdb_family_id) AS citing_families
    FROM tls228_docdb_fam_citn c
    WHERE c.cited_docdb_family_id = {FAMILY_ID}
    """)
    n_cites = int(citing["citing_families"].iloc[0]) if len(citing) else 0

    ipcs = q(f"""
    SELECT DISTINCT SUBSTR(ipc_class_symbol, 1, 4) AS subclass
    FROM tls209_appln_ipc WHERE appln_id IN ({fam_ids})
    """)
    subclasses = sorted(ipcs["subclass"].dropna().unique())
    neighbourhood = q(f"""
    SELECT COUNT(DISTINCT a.docdb_family_id) AS families
    FROM tls201_appln a
    JOIN tls209_appln_ipc i ON i.appln_id = a.appln_id
    WHERE SUBSTR(i.ipc_class_symbol, 1, 4) IN ({','.join(repr(s) for s in subclasses)})
      AND a.earliest_filing_year BETWEEN 2014 AND 2024
    """)
    n_neigh = int(neighbourhood["families"].iloc[0]) if len(neighbourhood) else 0
    return n_cites, n_neigh, subclasses

try:
    n_cites, n_neigh, subclasses = context_only()
    for qid, note in [
        ("B1", f"{n_cites} citing families - citations measure attention, not uniqueness. "
               f"Score left as the adviser set it"),
        ("B2", f"{n_cites} citing families - says nothing about technical superiority. "
               f"Score left as the adviser set it"),
        ("C4", f"{n_neigh:,} families filed 2014-2024 in {', '.join(subclasses)} - "
               f"context for how crowded the field is, not an answer"),
    ]:
        evidence[qid] = Answer(first_pass[qid], provenance="judgement", evidence=note)
        print(f"  {qid}  {first_pass[qid]} -> {first_pass[qid]}  [judgement, context attached]\n      {note}")
except Exception as exc:
    print(f"  B1/B2/C4 context unavailable: {type(exc).__name__}: {exc}")

  A4  3 -> 4  [informed]  (guess was 3)
      19 claims in the granted EP B1 vs a mean of 11.5 for granted EP - a proxy for breadth, not a measure of it


  B1  4 -> 4  [judgement, context attached]
      30 citing families - citations measure attention, not uniqueness. Score left as the adviser set it
  B2  4 -> 4  [judgement, context attached]
      30 citing families - says nothing about technical superiority. Score left as the adviser set it
  C4  3 -> 3  [judgement, context attached]
      2,252,525 families filed 2014-2024 in C12M, C12P, C12Q, G01N, G06T - context for how crowded the field is, not an answer


---

## Step 9 — The diff: what was guessed, what the record says

Now the comparison this notebook exists for. For every question we touched: the adviser's first
pass, what the data says, the fact behind it — and **what the correction is worth in euros**.

That last column is computed with `kit.npv_from_answers`, one question at a time. Expect a
column of zeros, and read it as the finding rather than as a bug: none of the eleven reachable
questions is one of the eight that carry money.

**The valuation does not move. The picture of the patent does.**

In [9]:
corrected = dict(example["answers"])
corrected.update(evidence)

base_npv = kit.npv_from_answers(example["financials"], example["answers"], spec)

rows = []
for qid in sorted(evidence, key=lambda x: (x[0], int(x[1:]))):
    ans = evidence[qid]
    trial = dict(example["answers"]); trial[qid] = ans
    rows.append({
        "Q": qid,
        "What it asks": spec[qid].factor,
        "Guessed": first_pass[qid],
        "Record says": ans.score,
        "Moved": "" if ans.score == first_pass[qid] else ("UP" if ans.score > first_pass[qid] else "DOWN"),
        "Provenance": ans.provenance,
        "Carries money": spec[qid].carries_money,
        "NPV effect (EUR)": kit.npv_from_answers(example["financials"], trial, spec) - base_npv,
        "Evidence": ans.evidence,
    })
diff = pd.DataFrame(rows)

new_npv = kit.npv_from_answers(example["financials"], corrected, spec)
prof_before = kit.profile(example["answers"], spec)
prof_after = kit.profile(corrected, spec)

print(f"answers touched      {len(evidence)} of 40"
      + (f"   ({len(failures)} query failures: {', '.join(failures)})" if failures else ""))
print(f"answers changed      {(diff['Guessed'] != diff['Record says']).sum()}")
print(f"provenance           {prof_after.provenance_counts}")
print(f"score                {prof_before.total_points} -> {prof_after.total_points} / 200")
print(f"average risk         {prof_before.average_risk:+.2f} -> {prof_after.average_risk:+.2f}")
print(f"\nNet Present Value    {base_npv:,.0f} -> {new_npv:,.0f} EUR"
      f"   (difference: {new_npv - base_npv:+,.0f})")

diff.drop(columns=["Evidence"])

answers touched      11 of 40
answers changed      5
provenance           {'measured': 2, 'informed': 6, 'judgement': 32}
score                138 -> 139 / 200
average risk         -0.39 -> -0.39

Net Present Value    1,248,870 -> 1,248,870 EUR   (difference: +0)


,Q,What it asks,Guessed,Record says,Moved,Provenance,Carries money,NPV effect (EUR)
0,A1,Patent status,4,5,UP,measured,False,0.0
1,A3,Patent term remaining,4,3,DOWN,measured,False,0.0
2,A4,Breadth of claim,3,4,UP,informed,False,0.0
3,A5,Geographical coverage,3,3,,informed,False,0.0
4,A7,Legal proceedings,3,3,,informed,False,0.0
5,B1,Unique technology,4,4,,judgement,False,0.0
6,B2,Substitute technology,4,4,,judgement,False,0.0
7,C4,Competitive/substitute products,3,3,,judgement,False,0.0
8,E1,Securing existing markets,4,5,UP,informed,False,0.0
9,E2,Winning new markets,3,2,DOWN,informed,False,0.0


In [10]:
fig = go.Figure()
moved = diff[diff["Guessed"] != diff["Record says"]]
labels = [f"{row['Q']} &#183; {row['What it asks']}" for _, row in moved.iterrows()]

fig.add_trace(go.Bar(
    y=labels, x=moved["Guessed"], orientation="h", name="the adviser guessed",
    marker={"color": kit.PALETTE["inactive"]},
))
fig.add_trace(go.Bar(
    y=labels, x=moved["Record says"], orientation="h", name="the record says",
    marker={"color": kit.PALETTE["efficiency"]},
))
fig.update_layout(
    **kit.CHART_LAYOUT, barmode="group", height=90 + 60 * max(len(moved), 1),
    title="Where the record disagrees with the guess - and none of it moves the money",
)
fig.update_xaxes(title="score (1-5)", dtick=1, range=[0, 5.4])
fig.update_yaxes(title=None, automargin=True)
fig.show()

for _, row in moved.iterrows():
    print(f"{row['Q']}: {row['Evidence']}")

A1: granted, and the opposition period expired with no opposition filed (26N, effective 2018-10-11)
A3: earliest filing 2014-06-13, nominal expiry 2034-06-13 = 7.8 years left; renewals paid to year 11 in DE, FR, GB, SE
A4: 19 claims in the granted EP B1 vs a mean of 11.5 for granted EP - a proxy for breadth, not a measure of it
E1: 7 of 8 jurisdictions in this family (AU, CA, CN, EP, KR, US, WO) are ones Q-LINEA had already filed in
E2: 1 of 8 jurisdictions are new for Q-LINEA (JP)


---

## Step 10 — Hand the corrected answers to notebooks 3 and 4

Two hand-offs, both through `ipscore_kit`:

1. **`evidence_answers.json`** — the full forty, with provenance and evidence. `kit.load_answers()`
   prefers this file over `worked_example.json` as soon as it exists, so notebooks 3 and 4 pick
   it up with no edit. Until this notebook has run, they fall back to the first pass and *say so*
   in the report.
2. **One report section** at `order` 450 — right after *"how much of this valuation can data
   reach?"*, which is exactly where a reader will ask *"so what did it reach?"*

Then **re-run notebooks 3 and 4**. The report's provenance panel stops reading
`0 measured · 0 informed · 40 judgement` — and its Net Present Value stays exactly where it was.

In [11]:
OUTPUT_DIR = Path("2_evidence_from_patstat_output")
OUTPUT_DIR.mkdir(exist_ok=True)

payload = {
    "label": (f"measured against PATSTAT for {example['patent']['publication']} "
              f"by 2_evidence_from_patstat.ipynb"),
    "patent": example["patent"],
    "queries_failed": failures,
    "answers": {qid: {"score": a.score, "provenance": a.provenance, "evidence": a.evidence}
                for qid, a in corrected.items()},
}
(OUTPUT_DIR / "evidence_answers.json").write_text(
    json.dumps(payload, ensure_ascii=False, indent=1), encoding="utf-8")

fragment = fig.to_html(full_html=False, include_plotlyjs=False,
                       div_id="fig_evidence", default_width="100%")

kit.record_section(
    450, "evidence", "What the record actually says",
    fragment_html=fragment,
    note=("Eleven of the forty answers were checked against PATSTAT. None of them is one of the "
          "eight that carry money, so the valuation is unchanged - only the picture of the "
          "patent is."),
    sheets={"evidence": diff.to_dict("records")},
    output_dir=OUTPUT_DIR,
)

answers_now, label = kit.load_answers()
print(f"wrote {OUTPUT_DIR}/evidence_answers.json")
print(f"kit.load_answers() now reports: {label}")
print("\nRe-run 3_valuation_and_scenarios.ipynb and 4_assemble_tool.ipynb to pick this up.")

wrote 2_evidence_from_patstat_output/evidence_answers.json
kit.load_answers() now reports: measured against PATSTAT for EP3074539B1 by 2_evidence_from_patstat.ipynb

Re-run 3_valuation_and_scenarios.ipynb and 4_assemble_tool.ipynb to pick this up.


---

## Done — and what it was worth

Eleven of the forty answers were checked against the public record, and several of them moved.
The term is arithmetic and the arithmetic is unforgiving. The geographic picture turns out to be
a different shape than the designation list suggests — designated in dozens of EPC states, in
force in a handful, with the real reach coming from the national grants outside Europe.

**And the Net Present Value did not move by a single euro.**

Both halves of that sentence are the module. A patent valuation has a checkable part and a
decisive part, and they do not overlap. The eight answers that carry money — years to market,
market growth, life expectancy, extra turnover, maintainability, development cost, production
cost, investment intensity — are forecasts about a market's future, and no database has an
opinion about the future.

So use the evidence layer for what it is good for:

* **Do not be wrong about the record.** It is free to check and embarrassing to get wrong.
* **Make the valuation auditable.** Every `measured` answer now carries a date and an event code.
* **Point at the guessing.** With eight movers, none of them measurable, everyone in the room
  can see what kind of object a valuation is — and that is a more useful thing to leave a
  workshop with than a number.

Next: re-run **`3_valuation_and_scenarios.ipynb`** and **`4_assemble_tool.ipynb`**.